# Exercises XP: Day 3 - BERT in Practice
Follow the prompts below. Replace each TODO marker with your own code or explanation before executing the cell.


## What you'll learn
- How to tokenize text with BERT and understand special tokens.
- How to run a pretrained sentiment pipeline.
- How to build custom BERT-based sentiment and NER analyzers.
- How to compare encoder (BERT) versus decoder (GPT) families.
- How BERT supplies retrieval power inside a RAG stack.


## What you will create
- A fully tokenized sentence with visible IDs and special tokens.
- A working sentiment pipeline powered by a fine-tuned DistilBERT model.
- Custom helper classes for sentiment classification and NER.
- A comparison table that contrasts BERT and GPT.
- A written explanation of how BERT embeddings drive retrieval in RAG.


> Mandatory preparation: watch "PyTorch in 100 Seconds" so the tensor outputs below feel intuitive.

## Exercise 1 - Tokenization with BERT
Objective: Explore how the bert-base-uncased tokenizer prepares text for model input.

Instructions:
1. (Optional) Install the required libraries.
2. Load the tokenizer, craft a sample sentence, and encode it with padding plus truncation.
3. Print the tokens next to their integer IDs and flag the special tokens.
4. Inspect the attention mask to see how padding is hidden from the model.

Deliverables:
- TODO: Provide the printed list of tokens and IDs with [CLS]/[SEP]/[PAD] highlighted.
- TODO: Document the padding choice you made and why it fits the sentence length.


In [ ]:
# Optional setup: install dependencies if they are missing in your environment.
# %pip install -q transformers torch


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

sample_sentence = "The quick brown fox jumps over the lazy dog near the river bank."
print(sample_sentence)

In [ ]:
encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=32,   # 14 real tokens + [CLS] + [SEP] = 16 → padded to 32
    return_attention_mask=True,
    return_tensors="pt"
)

input_ids = encoding["input_ids"][0].tolist()
tokens    = tokenizer.convert_ids_to_tokens(input_ids)

print("index | token        | id     | type")
print("--------------------------------------")
for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
    tag = ""
    if token == "[CLS]":
        tag = "← classification token (start)"
    elif token == "[SEP]":
        tag = "← separator token (end)"
    elif token == "[PAD]":
        tag = "← padding token (ignored by model)"
    print(f"{idx:>5} | {token:<12} | {token_id:>6}  {tag}")

print("\nAttention mask:", encoding["attention_mask"][0].tolist())
special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
print("Special tokens (index, token):", special_positions)

### Exercise 1 reflection

**[CLS] and [SEP] inside the encoder:**  
`[CLS]` (ID 101) is always the very first token. After all 12 transformer layers, its hidden state accumulates a global representation of the entire sequence — this vector is what downstream classification heads read. `[SEP]` (ID 102) marks the boundary between sentences (or the end of a single sentence). In two-sentence tasks (e.g., NLI, QA) a second `[SEP]` separates the two segments, and `token_type_ids` tell the encoder which segment each token belongs to.

**How the attention mask hides padding:**  
The attention mask is a binary vector (1 = real token, 0 = padding). Before computing scaled dot-product attention scores, the model adds a large negative value (≈ −10 000) to every position where the mask is 0. After the softmax, those positions receive ≈ 0 weight, so padded tokens contribute nothing to the final contextual representations.

## Exercise 2 - Sentiment analysis pipeline
Objective: Use a pretrained DistilBERT sentiment pipeline to classify a sentence.

Instructions:
1. Import the `pipeline` helper from transformers.
2. Build a pipeline that loads `distilbert-base-uncased-finetuned-sst-2-english`.
3. Pass in a sentence and review the predicted label and score.

Deliverables:
- TODO: Record the sentence you tested.
- TODO: Capture the label plus confidence score and interpret the result.


In [ ]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

sentence = "I absolutely loved the movie — the performances were outstanding and the story was gripping!"
prediction = sentiment_pipeline(sentence)
print(f"Sentence  : {sentence}")
print(f"Label     : {prediction[0]['label']}")
print(f"Confidence: {prediction[0]['score']:.4f}")
prediction

### Exercise 2 reflection

**Does the predicted label match expectations?**  
Yes. The sentence uses strongly positive words ("absolutely loved", "outstanding", "gripping"), so a `POSITIVE` prediction is expected. The model was fine-tuned on SST-2 (Stanford Sentiment Treebank), which is built from movie reviews — exactly this domain.

**What does the confidence score tell us?**  
A score close to 1.0 (e.g., 0.9998) means the model's softmax output is almost entirely concentrated on the POSITIVE class. Concretely, the raw logit for POSITIVE is much larger than for NEGATIVE, so after softmax almost all probability mass goes to POSITIVE. A lower score (e.g., 0.65) would signal an ambiguous or mixed sentence where the model is uncertain.

## Exercise 3 - Custom sentiment analyzer class
Objective: Rebuild the pipeline manually so you control tokenization, tensors, and scoring.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForSequenceClassification`.
2. Implement `BERTSentimentAnalyzer` with methods for initialization, preprocessing, and prediction.
3. Test the class with multiple sentences.

Hints:
- Keep a `max_length` attribute so you can reuse it while tokenizing.
- Apply `torch.softmax` to transform logits into probabilities.
- Return both the label and the probability for clarity.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Dict
import re

class BERTSentimentAnalyzer:
    def __init__(
        self,
        model_name: str = "distilbert-base-uncased-finetuned-sst-2-english",
        max_length: int = 128
    ):
        self.max_length = max_length
        self.device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer  = AutoTokenizer.from_pretrained(model_name)
        self.model      = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()
        # Map index → human-readable label from the model config
        self.id2label = self.model.config.id2label

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
        # Basic cleaning: strip extra whitespace
        text = re.sub(r"\s+", " ", text).strip()
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt"
        )
        # Move tensors to the right device
        return {k: v.to(self.device) for k, v in encoding.items()}

    def predict(self, text: str) -> Dict[str, float]:
        inputs = self.preprocess(text)
        with torch.no_grad():
            logits = self.model(**inputs).logits          # shape: (1, num_labels)
        probabilities = torch.softmax(logits, dim=-1)[0]  # shape: (num_labels,)
        predicted_idx = probabilities.argmax().item()
        return {
            "label":       self.id2label[predicted_idx],
            "probability": round(probabilities[predicted_idx].item(), 4),
            "scores":      {self.id2label[i]: round(p.item(), 4)
                            for i, p in enumerate(probabilities)}
        }

In [ ]:
analyzer = BERTSentimentAnalyzer()

samples = [
    "This restaurant is absolutely fantastic — best meal I've had in years!",
    "The service was terrible and the food was cold and tasteless.",
    "The product is okay, nothing special but it gets the job done.",
    "I can't believe how disappointing this experience was.",
]

print(f"{'Sentence':<55} | {'Label':<10} | {'Confidence'}")
print("-" * 85)
for text in samples:
    result = analyzer.predict(text)
    short  = text[:53] + ".." if len(text) > 55 else text
    print(f"{short:<55} | {result['label']:<10} | {result['probability']:.4f}")

## Exercise 4 - BERT for Named Entity Recognition
Objective: Build a lightweight class that runs a token-classification model and maps tokens to entity labels.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForTokenClassification`.
2. Implement `BERTNamedEntityRecognizer` with init plus a `recognize` method.
3. Tokenize sample text, run the model, convert the predictions to entity spans, and test with a short paragraph.

Deliverables:
- TODO: Return a list of dictionaries like `{text, entity, start, end}` for each detected entity.
- TODO: Explain how you handled subword tokens that begin with `##`.


In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        self.device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model     = AutoModelForTokenClassification.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()
        self.id2label  = self.model.config.id2label  # e.g. {0:"O", 1:"B-PER", ...}

    def recognize(self, text: str):
        # Tokenize with offset mapping so we can recover character positions
        encoding = self.tokenizer(
            text,
            return_tensors="pt",
            return_offsets_mapping=True,
            truncation=True,
            max_length=512
        )
        offset_mapping = encoding.pop("offset_mapping")[0].tolist()
        inputs = {k: v.to(self.device) for k, v in encoding.items()}

        with torch.no_grad():
            logits = self.model(**inputs).logits        # (1, seq_len, num_labels)

        predictions = logits[0].argmax(dim=-1).tolist() # one label per token
        tokens      = self.tokenizer.convert_ids_to_tokens(encoding["input_ids"][0].tolist())

        entities  = []
        current   = None   # dict tracking the entity being built

        for token, pred_id, (start, end) in zip(tokens, predictions, offset_mapping):
            label = self.id2label[pred_id]

            # Skip special tokens ([CLS], [SEP]) — their offsets are (0,0)
            if start == 0 and end == 0:
                if current:
                    entities.append(current)
                    current = None
                continue

            if label.startswith("B-"):
                # Begin a new entity
                if current:
                    entities.append(current)
                current = {
                    "text":   text[start:end],
                    "entity": label[2:],   # strip the "B-" prefix
                    "start":  start,
                    "end":    end
                }
            elif label.startswith("I-") and current and current["entity"] == label[2:]:
                # Continue the current entity.
                # Subword tokens starting with "##" are part of the previous word;
                # we extend the span without adding a space.
                if token.startswith("##"):
                    current["text"] += text[start:end]
                else:
                    current["text"] += " " + text[start:end]
                current["end"] = end
            else:
                # "O" label or mismatched I- tag — close any open entity
                if current:
                    entities.append(current)
                    current = None

        if current:
            entities.append(current)

        return entities

In [ ]:
ner = BERTNamedEntityRecognizer()

sample_text = (
    "Elon Musk, the CEO of Tesla and SpaceX, visited Paris last Tuesday "
    "to meet with Emmanuel Macron at the Élysée Palace. "
    "The two discussed a potential partnership between SpaceX and the European Space Agency."
)

entities = ner.recognize(sample_text)

print(f"{'Entity text':<35} | {'Type':<6} | start | end")
print("-" * 60)
for e in entities:
    print(f"{e['text']:<35} | {e['entity']:<6} | {e['start']:>5} | {e['end']:>3}")

print(f"\nTotal entities found: {len(entities)}")

## Exercise 5 - Comparing BERT and GPT

| Category | BERT | GPT |
|----------|------|-----|
| **Architecture** | Encoder-only transformer — sees the full context (left **and** right) simultaneously via bidirectional self-attention | Decoder-only transformer — processes tokens left-to-right with causal (masked) self-attention; cannot look ahead |
| **Primary purpose** | Language *understanding*: learns rich contextual representations of existing text | Language *generation*: predicts the next token to produce fluent, coherent new text |
| **Typical use cases** | Text classification, NER, question answering, sentence similarity, NLI | Text generation, dialogue systems, summarisation, code completion, creative writing |
| **Strengths** | Deep bidirectional context gives superior understanding; highly effective when fine-tuned on small labelled datasets | Scales extremely well to very large sizes; zero/few-shot generation without task-specific labels |
| **Weaknesses** | Cannot generate text natively; requires labelled data for most downstream tasks; slower at inference on long sequences | Tends to hallucinate; unidirectional context is less powerful for pure understanding tasks; expensive to fine-tune |

**Key insight:** choose BERT (or its variants) when you need to *classify*, *extract*, or *compare* text; choose GPT-family models when you need to *produce* new text.

## Exercise 6 - BERT inside Retrieval-Augmented Generation

### 1. How BERT encodes queries and documents
BERT processes a piece of text through 12 (or 24) transformer layers and produces a hidden-state vector for every token. The standard practice for retrieval is to take the `[CLS]` token's final hidden state — a single 768-dimensional vector — as a dense embedding that captures the *semantic meaning* of the entire passage. Importantly, because BERT is bidirectional, this embedding reflects the full context: the word "bank" near "river" produces a different vector than "bank" near "money". Both the user query *and* every document in the corpus are encoded this way, mapping them into the same dense vector space.

### 2. Storing and searching embeddings in a vector database
Once encoded, each document embedding is stored in a **vector database** (e.g., FAISS, Pinecone, Weaviate, Chroma). These systems index dense vectors so that nearest-neighbour search — finding the *k* documents whose embeddings are most similar (via cosine or dot-product similarity) to the query embedding — runs in milliseconds even over millions of documents. At query time, the user's question is encoded with the same BERT model, and the vector DB returns the top-*k* most semantically similar passages.

### 3. Handing retrieved passages to a generative model
The top-*k* passages are concatenated with the original query to form a **context-augmented prompt**: `[Retrieved passage 1] ... [Retrieved passage k] \n\n Question: {query} \n Answer:`. This enriched prompt is then fed to a generative model (GPT, Llama, Mistral…). Because the LLM now has relevant factual context in its input window, it can ground its answer in real, up-to-date information rather than relying solely on parameters trained months or years ago — dramatically reducing hallucination.

### 4. Concrete example — Enterprise customer support
A SaaS company indexes its entire knowledge base (product docs, past tickets, release notes) using BERT embeddings stored in a vector DB. When a customer asks *"Why does the export to PDF fail on tables with merged cells?"*, BERT encodes the query and retrieves the 3 most relevant documentation sections. Those sections plus the question are passed to GPT-4, which writes a clear, accurate answer. Without the retrieval step, GPT-4 would either hallucinate product-specific details or answer generically. BERT-powered RAG makes the system accurate **and** maintainable: updating a doc re-encodes only that chunk, with no model retraining required.